# 🎓 Module 10 — Dataset Lab: prepare data for fine-tuning
### Marhaba Cattle & Dairy Farm
**Designed by Engr Usman Hasan** · Bano Qabil Generative AI course

**Aaj ka mission:** apne project ke liye ek *fine-tuning dataset* banana, jo agle hafte model train karne mein use hoga.

| Step | Kya karenge |
|---|---|
| 1 | Tools banana (`make_dataset.py`, `validate_dataset.py`) |
| 2 | Ek tayyar example dataset dekhna (farm order slips) |
| 3 | CSV → `train / val / test.jsonl` |
| 4 | 🎮 Game: kharab dataset mein ghaltiyan dhoondna |
| 5 | **Apna dataset** banana, check karna aur download karna |

No GPU, no API key. Har cell `Shift + Enter` se chalayein.

## ⚙️ Step 1 — Create the two tools
Yeh wahi files hain jo VS Code folder mein hain. Har line par comment hai, khol kar parhein!

In [ ]:
%%writefile make_dataset.py
"""
make_dataset.py  -  Module 10: turn a simple spreadsheet into a fine-tuning dataset
====================================================================================
Designed by Engr Usman Hasan  (Bano Qabil Generative AI course)

WHY THIS FILE EXISTS
  Writing JSONL by hand is slow and easy to break (one missing quote = broken file).
  So students write their examples in Excel / Google Sheets with TWO columns:

      input   |  output
      --------+---------------------------
      what the user says | the PERFECT answer

  ...then this script does the boring part.

THE FLOW
  your_folder/examples.csv  +  your_folder/system_prompt.txt
                     |
                     v
        1. read the system prompt (the same for every example)
        2. read every row of the CSV
        3. turn each row into one chat example (system / user / assistant)
        4. shuffle the examples (so the split is fair)
        5. split 80% train / 10% validation / 10% test
        6. write train.jsonl, val.jsonl, test.jsonl  (one example per line)

HOW TO RUN
  python make_dataset.py examples/01_farm_order_slips
  python make_dataset.py my_project            <- your own folder

No extra packages needed: only Python's built-in modules.
"""

# 'csv' reads spreadsheet files saved as .csv
import csv
# 'json' turns Python dictionaries into JSON text
import json
# 'os' builds file paths that work on Windows, Mac and Linux
import os
# 'random' shuffles the examples
import random
# 'sys' lets us read the folder name typed after the script name
import sys

# ---------------------------------------------------------------------
# SETTINGS  (change these if your teacher asks you to)
# ---------------------------------------------------------------------

# 80% of the examples go to training (the "textbook" the model studies)
TRAIN_SHARE = 0.80
# 10% go to validation (the "practice test" checked during training)
VAL_SHARE = 0.10
# the rest (about 10%) become the test set (the "final exam", kept sealed)
# a fixed seed means the shuffle gives the SAME result every time you run it
SEED = 42


# ---------------------------------------------------------------------
# STEP 1: read the system prompt
# ---------------------------------------------------------------------

# a function that reads system_prompt.txt from the folder
def read_system_prompt(folder):
    # build the full path, e.g. "examples/01_farm_order_slips/system_prompt.txt"
    path = os.path.join(folder, "system_prompt.txt")
    # if the file is missing, explain what to do and stop
    if not os.path.exists(path):
        # stop the program with a clear message
        sys.exit(f"Missing {path}. Create it and write ONE instruction for the model.")
    # open the file (utf-8 handles Urdu letters and emojis)
    with open(path, encoding="utf-8") as f:
        # read it and remove spaces/new lines at the start and end
        prompt = f.read().strip()
    # an empty system prompt is a mistake, so stop early
    if not prompt:
        # stop with a message
        sys.exit("system_prompt.txt is empty. Write the instruction for the model.")
    # give the prompt back
    return prompt


# ---------------------------------------------------------------------
# STEP 2: read the CSV rows
# ---------------------------------------------------------------------

# a function that reads examples.csv and returns a list of (input, output) pairs
def read_rows(folder):
    # full path to the CSV file
    path = os.path.join(folder, "examples.csv")
    # stop with a message if the file does not exist
    if not os.path.exists(path):
        # explain how to make the CSV
        sys.exit(f"Missing {path}. In Excel/Sheets: File > Download > CSV, columns 'input' and 'output'.")
    # the list we will fill
    rows = []
    # open the CSV; newline="" is the correct way to open CSV files in Python
    with open(path, encoding="utf-8-sig", newline="") as f:
        # DictReader uses the first row (the header) as column names
        reader = csv.DictReader(f)
        # check that the two needed columns exist
        if reader.fieldnames is None or "input" not in reader.fieldnames or "output" not in reader.fieldnames:
            # stop with a clear message
            sys.exit("The CSV must have a header row with the columns: input, output")
        # go through every row; line_no starts at 2 because line 1 is the header
        for line_no, row in enumerate(reader, start=2):
            # clean the input text (None becomes "")
            user_text = (row["input"] or "").strip()
            # clean the output text
            answer = (row["output"] or "").strip()
            # skip rows where one side is empty, and tell the student
            if not user_text or not answer:
                # print a warning with the line number so it is easy to fix
                print(f"  skipped line {line_no}: input or output is empty")
                # jump to the next row
                continue
            # keep this pair
            rows.append((user_text, answer))
    # give the list back
    return rows


# ---------------------------------------------------------------------
# STEP 3: turn one row into one chat example
# ---------------------------------------------------------------------

# build the dictionary that fine-tuning tools expect
def to_chat_example(system_prompt, user_text, answer):
    # every example is {"messages": [system, user, assistant]}
    return {
        # a list of three messages, in this exact order
        "messages": [
            # 1) the instruction: SAME in every example
            {"role": "system", "content": system_prompt},
            # 2) what the user/customer says
            {"role": "user", "content": user_text},
            # 3) the perfect answer: the model will learn to copy THIS
            {"role": "assistant", "content": answer},
        ]
    }


# ---------------------------------------------------------------------
# STEP 5 + 6: split and save
# ---------------------------------------------------------------------

# write a list of examples into a .jsonl file (one JSON object per line)
def write_jsonl(path, examples):
    # open the file for writing ("w" replaces any old file)
    with open(path, "w", encoding="utf-8") as f:
        # one example at a time
        for example in examples:
            # ensure_ascii=False keeps Urdu letters and emojis readable
            line = json.dumps(example, ensure_ascii=False)
            # write the JSON plus a new line, so each example is on its own line
            f.write(line + "\n")


# the main function: runs all the steps in order
def main():
    # the folder must be given after the script name
    if len(sys.argv) < 2:
        # show how to use the script, then stop
        sys.exit("Usage: python make_dataset.py <folder>   e.g.  python make_dataset.py my_project")
    # the folder name the student typed
    folder = sys.argv[1]

    # STEP 1: the system prompt
    system_prompt = read_system_prompt(folder)
    # STEP 2: the CSV rows
    rows = read_rows(folder)
    # we need at least 10 rows to make three files
    if len(rows) < 10:
        # stop with a message
        sys.exit(f"Only {len(rows)} usable rows. Write at least 10 (homework target: 200+).")

    # STEP 3: convert every row into a chat example
    examples = [to_chat_example(system_prompt, u, a) for u, a in rows]

    # STEP 4: shuffle with a fixed seed so the result is repeatable
    random.Random(SEED).shuffle(examples)

    # STEP 5: work out how many examples go into each file
    total = len(examples)
    # number for validation: at least 1
    n_val = max(1, round(total * VAL_SHARE))
    # number for test: at least 1, and roughly the same as validation
    n_test = max(1, total - round(total * TRAIN_SHARE) - n_val)
    # training gets everything that is left
    n_train = total - n_val - n_test
    # cut the shuffled list into three parts
    train = examples[:n_train]
    # the next n_val examples
    val = examples[n_train:n_train + n_val]
    # the last n_test examples
    test = examples[n_train + n_val:]

    # STEP 6: save the three files next to the CSV
    write_jsonl(os.path.join(folder, "train.jsonl"), train)
    # validation file
    write_jsonl(os.path.join(folder, "val.jsonl"), val)
    # test file (do NOT look at it while improving your data!)
    write_jsonl(os.path.join(folder, "test.jsonl"), test)

    # print a friendly summary
    print(f"\nDone! {total} examples from {folder}/examples.csv")
    # show the size of each file
    print(f"  train.jsonl : {len(train)}  (the textbook)")
    # validation size
    print(f"  val.jsonl   : {len(val)}  (the practice test)")
    # test size
    print(f"  test.jsonl  : {len(test)}  (the sealed final exam)")
    # remind the student of the next step
    print(f"\nNext: python validate_dataset.py {folder}")


# run main() only when this file is started directly (not when imported)
if __name__ == "__main__":
    # start the program
    main()


In [ ]:
%%writefile validate_dataset.py
"""
validate_dataset.py  -  Module 10: check a fine-tuning dataset BEFORE training
===============================================================================
Designed by Engr Usman Hasan  (Bano Qabil Generative AI course)

WHY THIS FILE EXISTS
  Training on a broken dataset wastes GPU time and teaches the model bad habits.
  This script is the "dataset doctor": it reads your .jsonl files and lists
  every problem it can find, with the LINE NUMBER, so you can fix it.

WHAT IT CHECKS
  Errors (must fix):                    Warnings (please look):
  - a line is not valid JSON            - different system prompts
  - "messages" list is missing          - the same input appears twice
  - wrong role order / empty text       - looks like a real phone / email / CNIC
  - last message is not "assistant"     - very long examples
  - JSON answers that do not parse      - fewer than 100 examples
  - JSON answers with different keys    - unbalanced labels
  - a test example also in train        - chatty answers ("Sure! Here is...")

  The script checks FORMAT.  Only a human can check MEANING
  (is the answer actually correct?).  Do both!

HOW TO RUN
  python validate_dataset.py examples/01_farm_order_slips      <- a folder (train/val/test)
  python validate_dataset.py examples/04_bad_dataset/bad_examples.jsonl   <- one file

No extra packages needed: only Python's built-in modules.
"""

# 'json' reads each line of the .jsonl file
import json
# 'os' checks whether a path is a folder or a file
import os
# 're' (regular expressions) finds patterns like phone numbers
import re
# 'sys' reads the path typed after the script name
import sys
# Counter counts things, e.g. how many times each label appears
from collections import Counter

# ---------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------

# homework minimum: fewer examples than this gives a warning
MIN_EXAMPLES = 100
# examples longer than this (roughly, in tokens) give a warning
MAX_TOKENS = 512
# patterns that look like private data (Pakistani mobile, email, CNIC)
PRIVATE_PATTERNS = {
    # e.g. 0321-4567890 or 03214567890 or +92 321 4567890
    "phone number": re.compile(r"(\+92[\s-]?|0)3\d{2}[\s-]?\d{7}"),
    # anything like name@site.com
    "email address": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    # CNIC format 42101-1234567-1
    "CNIC number": re.compile(r"\b\d{5}-\d{7}-\d\b"),
}
# answers that start like a chatbot instead of giving the clean answer
CHATTY_STARTS = ("sure", "here is", "here's", "of course", "certainly")


# ---------------------------------------------------------------------
# small helpers
# ---------------------------------------------------------------------

# a rough token count: about 4 characters per token in English
def approx_tokens(text):
    # integer division; +1 so short texts never count as 0
    return len(text) // 4 + 1


# try to read text as a JSON object; return the dict, or None if it fails
def as_json_object(text):
    # 'try' = attempt it without crashing
    try:
        # turn the text into Python data
        value = json.loads(text)
    # json.loads raises ValueError for broken JSON
    except ValueError:
        # not valid JSON
        return None
    # we only accept a JSON object { ... }, not a list or a number
    return value if isinstance(value, dict) else None


# ---------------------------------------------------------------------
# read and check ONE file
# ---------------------------------------------------------------------

# returns a list of good examples plus lists of errors and warnings
def check_file(path):
    # problems that MUST be fixed
    errors = []
    # problems that SHOULD be looked at
    warnings = []
    # examples that passed the structure checks
    good = []
    # the short file name, used in messages
    name = os.path.basename(path)

    # open the file (utf-8 for Urdu letters and emojis)
    with open(path, encoding="utf-8") as f:
        # read all lines into a list
        lines = f.readlines()

    # go through every line; line numbers start at 1 like in VS Code
    for no, raw in enumerate(lines, start=1):
        # remove the new-line character and spaces at the ends
        raw = raw.strip()
        # skip completely empty lines (but tell the student)
        if not raw:
            # a warning, not an error
            warnings.append(f"{name} line {no}: empty line (delete it)")
            # go to the next line
            continue
        # CHECK 1: is the line valid JSON?
        try:
            # parse the line
            example = json.loads(raw)
        # broken JSON lands here
        except ValueError as e:
            # record the error with Python's explanation
            errors.append(f"{name} line {no}: not valid JSON ({e.msg})")
            # cannot check anything else on this line
            continue
        # CHECK 2: is there a "messages" list?
        messages = example.get("messages") if isinstance(example, dict) else None
        # missing or not a list = error
        if not isinstance(messages, list) or not messages:
            # record the error
            errors.append(f"{name} line {no}: no \"messages\" list")
            # next line
            continue
        # CHECK 3: every message needs a role and non-empty text
        roles = []
        # a flag that turns False when something is wrong
        ok = True
        # look at each message in the example
        for m in messages:
            # the role, e.g. "user"
            role = m.get("role") if isinstance(m, dict) else None
            # the text of the message
            content = m.get("content") if isinstance(m, dict) else None
            # role must be one of the three allowed words
            if role not in ("system", "user", "assistant"):
                # record the error
                errors.append(f"{name} line {no}: unknown role {role!r}")
                # mark this example as broken
                ok = False
            # content must be a string with real text in it
            elif not isinstance(content, str) or not content.strip():
                # record the error
                errors.append(f"{name} line {no}: empty {role} message")
                # mark as broken
                ok = False
            # remember the role order
            roles.append(role)
        # CHECK 4: role order must be (system), user, assistant, user, assistant ...
        turns = roles[1:] if roles and roles[0] == "system" else roles
        # the pattern we expect: user, assistant, user, assistant ...
        expected = ["user", "assistant"] * (len(turns) // 2 + 1)
        # compare the real order with the expected order
        if ok and (turns != expected[:len(turns)] or not turns or turns[-1] != "assistant"):
            # record the error and show the actual order
            errors.append(f"{name} line {no}: role order {roles} (must end with assistant)")
            # mark as broken
            ok = False
        # only keep examples that passed checks 1-4
        if ok:
            # store the example together with its line number
            good.append((no, messages))
    # give everything back
    return good, errors, warnings


# ---------------------------------------------------------------------
# checks that look at ALL examples of a file together
# ---------------------------------------------------------------------

# the system prompt, user text and answer of one example
def parts(messages):
    # the system message text, or "" if there is none
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    # the first user message
    user = next(m["content"] for m in messages if m["role"] == "user")
    # the LAST assistant message (the answer being taught)
    answer = [m["content"] for m in messages if m["role"] == "assistant"][-1]
    # return all three
    return system, user, answer


# look for patterns across all examples of one file
def check_together(name, good, errors, warnings):
    # nothing to compare if there are no good examples
    if not good:
        # stop here
        return
    # CHECK 5: is the system prompt the same everywhere?
    systems = Counter(parts(m)[0] for _, m in good)
    # more than one different system prompt = warning
    if len(systems) > 1:
        # the most common one is probably the "right" one
        main_prompt = systems.most_common(1)[0][0]
        # list the lines that use a different prompt
        odd = [no for no, m in good if parts(m)[0] != main_prompt]
        # record the warning
        warnings.append(f"{name}: {len(systems)} different system prompts (odd ones on lines {odd})")
    # CHECK 6: duplicate inputs
    seen = {}
    # go through the examples in order
    for no, m in good:
        # the user text, in lowercase so "Hi" and "hi" count as the same
        key = parts(m)[1].strip().lower()
        # already seen this input before?
        if key in seen:
            # record the warning with both line numbers
            warnings.append(f"{name} line {no}: duplicate of line {seen[key]}")
        # remember first appearance
        else:
            # store the line number
            seen[key] = no
    # CHECK 7: is this a JSON task? (most answers are JSON objects)
    answers = [(no, parts(m)[2]) for no, m in good]
    # count answers that parse as JSON objects
    json_count = sum(1 for _, a in answers if as_json_object(a) is not None)
    # if more than half are JSON, we treat the task as "answer must be JSON"
    if json_count > len(answers) / 2:
        # the most common set of keys = the expected format
        key_sets = Counter(tuple(sorted(as_json_object(a))) for _, a in answers if as_json_object(a) is not None)
        # the winner
        expected_keys = key_sets.most_common(1)[0][0]
        # check every answer
        for no, a in answers:
            # try to parse it
            obj = as_json_object(a)
            # not JSON at all
            if obj is None:
                # record the error and show the start of the answer
                errors.append(f"{name} line {no}: answer is not clean JSON -> {a[:45]!r}")
            # JSON, but with different keys
            elif tuple(sorted(obj)) != expected_keys:
                # show which keys are extra or missing
                extra = set(obj) - set(expected_keys)
                # keys that should be there but are not
                missing = set(expected_keys) - set(obj)
                # record the error
                errors.append(f"{name} line {no}: different keys (extra {sorted(extra)}, missing {sorted(missing)})")
    # not a JSON task: warn about chatty answers instead
    else:
        # check each answer
        for no, a in answers:
            # does it start like "Sure! Here is..."?
            if a.strip().lower().startswith(CHATTY_STARTS):
                # record the warning
                warnings.append(f"{name} line {no}: chatty answer start -> {a[:30]!r}")
    # CHECK 8: private data
    for no, m in good:
        # join all text of the example
        text = " ".join(x["content"] for x in m)
        # try every private-data pattern
        for label, pattern in PRIVATE_PATTERNS.items():
            # found something?
            if pattern.search(text):
                # record the warning
                warnings.append(f"{name} line {no}: looks like a real {label}. Use fake data!")
    # CHECK 9: very long examples
    for no, m in good:
        # rough size of the whole example
        size = approx_tokens(" ".join(x["content"] for x in m))
        # too long?
        if size > MAX_TOKENS:
            # record the warning
            warnings.append(f"{name} line {no}: about {size} tokens (keep under {MAX_TOKENS} for free Colab)")


# ---------------------------------------------------------------------
# label balance (only for classification tasks)
# ---------------------------------------------------------------------

# print a small bar chart if the answers look like labels
def show_labels(all_good, warnings):
    # all answers
    answers = [parts(m)[2].strip() for _, m in all_good]
    # count each different answer
    counts = Counter(answers)
    # labels = short answers (max 3 words) and only a few different ones
    if not answers or len(counts) > 10 or any(len(a.split()) > 3 for a in answers):
        # not a classification task, nothing to show
        return
    # heading
    print("\nLabel balance:")
    # the biggest count, used to scale the bars
    biggest = max(counts.values())
    # one row per label, most common first
    for label, n in counts.most_common():
        # bar of '#' characters, max 30 long
        bar = "#" * max(1, round(30 * n / biggest))
        # print label, bar and count
        print(f"  {label:<12} {bar} {n}")
    # warn if the biggest label is more than 3x the smallest
    if biggest > 3 * min(counts.values()):
        # record the warning
        warnings.append("labels are unbalanced (one label is more than 3x another). Add examples of the small label")


# ---------------------------------------------------------------------
# main: decide folder vs file, run all checks, print the report
# ---------------------------------------------------------------------

# the main function
def main():
    # a path must be given
    if len(sys.argv) < 2:
        # show usage and stop
        sys.exit("Usage: python validate_dataset.py <folder or file.jsonl>")
    # the path the student typed
    target = sys.argv[1]
    # a folder: check train/val/test that exist inside it
    if os.path.isdir(target):
        # the three standard file names, in this order
        files = [os.path.join(target, n) for n in ("train.jsonl", "val.jsonl", "test.jsonl")]
        # keep only the ones that exist
        files = [f for f in files if os.path.exists(f)]
    # a single file
    else:
        # a list with one file
        files = [target]
    # no files found
    if not files:
        # explain and stop
        sys.exit("No .jsonl files found. Run make_dataset.py first.")

    # collected results
    all_errors, all_warnings, all_good, per_file = [], [], [], {}
    # check each file
    for path in files:
        # structure checks (1-4)
        good, errors, warnings = check_file(path)
        # pattern checks (5-9)
        check_together(os.path.basename(path), good, errors, warnings)
        # collect errors
        all_errors += errors
        # collect warnings
        all_warnings += warnings
        # collect good examples
        all_good += good
        # remember good examples per file name (for the leakage check)
        per_file[os.path.basename(path)] = good

    # CHECK 10: leakage - a test or val input that is also in train
    if "train.jsonl" in per_file:
        # all training inputs in lowercase
        train_inputs = {parts(m)[1].strip().lower() for _, m in per_file["train.jsonl"]}
        # look in val and test
        for other in ("val.jsonl", "test.jsonl"):
            # go through that file's examples
            for no, m in per_file.get(other, []):
                # same input also in train = the exam questions were in the textbook!
                if parts(m)[1].strip().lower() in train_inputs:
                    # record the error
                    all_errors.append(f"{other} line {no}: also in train.jsonl (leakage, the exam is not fair)")

    # CHECK 11: enough examples in total?
    if len(all_good) < MIN_EXAMPLES:
        # record the warning
        all_warnings.append(f"only {len(all_good)} good examples (homework target: {MIN_EXAMPLES}+, better 200+)")

    # REPORT: how many examples per file
    print("=" * 60)
    # title line
    print("DATASET CHECK:", target)
    # separator
    print("=" * 60)
    # one line per file
    for name, good in per_file.items():
        # number of good examples in that file
        print(f"  {name:<20} {len(good)} good examples")
    # label bar chart (classification only)
    show_labels(all_good, all_warnings)
    # print all errors
    print(f"\nERRORS ({len(all_errors)})  - must fix:")
    # one per line
    for e in all_errors:
        # red cross for errors
        print("  [X]", e)
    # print all warnings
    print(f"\nWARNINGS ({len(all_warnings)})  - please look:")
    # one per line
    for w in all_warnings:
        # exclamation mark for warnings
        print("  [!]", w)
    # final verdict
    if all_errors:
        # cannot train yet
        print("\nRESULT: NOT READY. Fix the errors and run this check again.")
    # warnings only
    elif all_warnings:
        # can train, but look first
        print("\nRESULT: READY, but read the warnings first.")
    # perfect
    else:
        # all good
        print("\nRESULT: READY for fine-tuning next week!")
    # remind them of the human check
    print("Remember: this script checks FORMAT. Read 20 random examples yourself to check MEANING.")


# run main() only when this file is started directly
if __name__ == "__main__":
    # start the program
    main()


In [ ]:
# Create the example files (so you don't need to upload anything)
# 'os' makes folders
import os

# file path -> file content
files = {
    'examples/01_farm_order_slips/examples.csv': 'input,output\nCan you send half kg of white butter on Friday? Address: clifton block 5,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Clifton Block 5""}"\n"2 kg butter please, deliver this evening to gulshan","{""is_order"": true, ""product"": ""butter"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Gulshan-e-Iqbal""}"\nDo you deliver in Malir?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nCan you send 4 litres of fresh milk day after tomorrow? Address: pechs,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": ""day after tomorrow"", ""area"": ""PECHS""}"\nplz 3 litre lassi PECHS,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 3, ""unit"": ""litre"", ""delivery_time"": null, ""area"": ""PECHS""}"\n"aaj shaam 1 kilo dahi rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": null}"\n"mujhe desi ghee chahiye parson, gulshan","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""day after tomorrow"", ""area"": ""Gulshan-e-Iqbal""}"\n"abhi ke liye 2 kilo desi ghee book kar lein, north nazimabad","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""now"", ""area"": ""North Nazimabad""}"\nfarm Sunday ko khula hota hai?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"mujhe doodh chahiye parson, bahadurabad","{""is_order"": true, ""product"": ""milk"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""day after tomorrow"", ""area"": ""Bahadurabad""}"\naaj ka doodh khatta tha,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nbhai parson 1 kilo makhan bhej dena pechs,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""PECHS""}"\n"kal shaam aadha kilo dahi rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": null}"\n"north nazimabad wala order: 1 kilo ghee, parson","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""North Nazimabad""}"\n"mujhe lassi chahiye parson, Gulshan e Iqbal","{""is_order"": true, ""product"": ""lassi"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""day after tomorrow"", ""area"": ""Gulshan-e-Iqbal""}"\n"Assalam o alaikum, Clifton blk 5 mein 1 kg dahi chahiye aaj shaam","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Clifton Block 5""}"\n"Sunday subah aadha kilo dahi rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""Sunday morning"", ""area"": null}"\nAoA bhai aadha kilo makkhan Friday ko chahiye. address Bahadurabad,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Bahadurabad""}"\n"Clifton blk 5 wala order: 2 kilo dahi, Friday ko","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Clifton Block 5""}"\nbhai kal shaam 4 litre doodh bhej dena Clifton blk 5,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": ""tomorrow evening"", ""area"": ""Clifton Block 5""}"\nCan you send 1 kg of butter on Friday? Address: Clifton blk 5,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Clifton Block 5""}"\n"kal subah ke liye 2 kg desi ghee book kar lein, bahadurabad","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Bahadurabad""}"\n"kal subah ke liye 2 liter doodh book kar lein, north nazimabad","{""is_order"": true, ""product"": ""milk"", ""quantity"": 2, ""unit"": ""litre"", ""delivery_time"": ""tomorrow morning"", ""area"": ""North Nazimabad""}"\n"parson ke liye 1 kg makhan book kar lein, north nazimabad","{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""North Nazimabad""}"\n"Hi, I need 2 kg dahi tomorrow morning at Bahadurabad","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Bahadurabad""}"\ndoodh ka rate kya hai?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nabhi dahi bhej dein pechs,"{""is_order"": true, ""product"": ""yogurt"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""now"", ""area"": ""PECHS""}"\nCan you send 5 kilos of dahi tomorrow morning? Address: PECHS,"{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""PECHS""}"\n"kal shaam ke liye aadha kilo makhan book kar lein, pechs","{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": ""PECHS""}"\ndelivery charges kitne hain DHA mein?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"north nazimabad wala order: 1 kg dahi, parson","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""North Nazimabad""}"\nbhai aaj shaam 5 kg makkhan bhej dena Clifton blk 5,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Clifton Block 5""}"\nbhai aaj shaam aadha kilo desi ghee bhej dena clifton block 5,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Clifton Block 5""}"\n"Assalam o alaikum, Gulshan e Iqbal mein 1 kg dahi chahiye Friday ko","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Gulshan-e-Iqbal""}"\n"abhi 4 ltr doodh rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""milk"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": ""now"", ""area"": null}"\nIs the milk boiled or raw?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"Assalam o alaikum, clifton block 5 mein 3 kilo desi ghee chahiye kal shaam","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": ""Clifton Block 5""}"\n10 liter doodh bhejwa dein PECHS please,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": null, ""area"": ""PECHS""}"\ndahi 1 kg parson tak pechs pohncha dein,"{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""PECHS""}"\n"clifton block 5 wala order: 1 kg ghee, aaj shaam","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Clifton Block 5""}"\n"abhi 3 liter lassi rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""lassi"", ""quantity"": 3, ""unit"": ""litre"", ""delivery_time"": ""now"", ""area"": null}"\nAoA bhai 4 liter doodh parson chahiye. address N. Nazimabad,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": ""day after tomorrow"", ""area"": ""North Nazimabad""}"\n"Hi, I need 1 litres fresh milk tomorrow evening at north nazimabad","{""is_order"": true, ""product"": ""milk"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""tomorrow evening"", ""area"": ""North Nazimabad""}"\nCan you send half kg of butter this evening? Address: gulshan,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Gulshan-e-Iqbal""}"\n"Assalam o alaikum, N. Nazimabad mein 1 kilo dahi chahiye abhi","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""now"", ""area"": ""North Nazimabad""}"\n"parson ke liye aadha kilo makkhan book kar lein, north nazimabad","{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""North Nazimabad""}"\n"Assalam o alaikum, N. Nazimabad mein 1 ltr dudh chahiye Friday ko","{""is_order"": true, ""product"": ""milk"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""Friday"", ""area"": ""North Nazimabad""}"\nplz 1 liter doodh gulshan,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": null, ""area"": ""Gulshan-e-Iqbal""}"\naadha kilo makkhan bhejwa dein DHA ph 6 please,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": null, ""area"": ""DHA Phase 6""}"\nbhai Friday ko 1 kg desi ghee bhej dena bahadurabad,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Bahadurabad""}"\n"Assalam o alaikum, dha phase6 mein 10 ltr doodh chahiye kal shaam","{""is_order"": true, ""product"": ""milk"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""tomorrow evening"", ""area"": ""DHA Phase 6""}"\nAoA bhai 5 ltr doodh kal subah chahiye. address north nazimabad,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 5, ""unit"": ""litre"", ""delivery_time"": ""tomorrow morning"", ""area"": ""North Nazimabad""}"\n"aaj shaam ke liye 1 kilo ghee book kar lein, Bahadurabad","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Bahadurabad""}"\n"shukriya bhai, order mil gaya","{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"Sunday subah ke liye 1 kilo dahi book kar lein, Clifton blk 5","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""Sunday morning"", ""area"": ""Clifton Block 5""}"\nCan you send 1 litres of lassi on Friday? Address: clifton block 5,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""Friday"", ""area"": ""Clifton Block 5""}"\nplz 5 litre lassi PECHS,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 5, ""unit"": ""litre"", ""delivery_time"": null, ""area"": ""PECHS""}"\n"pechs wala order: aadha kilo ghee, aaj shaam","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""PECHS""}"\nEid Mubarak to the whole Marhaba team!,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"kal subah aadha kilo makkhan rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": null}"\n"Hello, are you open today?","{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"1 liters fresh milk please, deliver this evening to Gulshan e Iqbal","{""is_order"": true, ""product"": ""milk"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""today evening"", ""area"": ""Gulshan-e-Iqbal""}"\n"Gulshan e Iqbal wala order: 2 kilo dahi, abhi","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""now"", ""area"": ""Gulshan-e-Iqbal""}"\nAoA bhai 3 kg desi ghee parson chahiye. address north nazimabad,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""North Nazimabad""}"\ndudh 10 liter parson tak Clifton blk 5 pohncha dein,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""day after tomorrow"", ""area"": ""Clifton Block 5""}"\n"5 kilos yogurt please, deliver this evening to PECHS","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""PECHS""}"\n"Bahadurabad wala order: 10 liter lassi, kal shaam","{""is_order"": true, ""product"": ""lassi"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""tomorrow evening"", ""area"": ""Bahadurabad""}"\n"Assalam o alaikum, gulshan mein 5 kg desi ghee chahiye aaj shaam","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Gulshan-e-Iqbal""}"\nAoA bhai 10 litre dudh kal subah chahiye. address N. Nazimabad,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""tomorrow morning"", ""area"": ""North Nazimabad""}"\n"kal subah ke liye 3 kg makkhan book kar lein, pechs","{""is_order"": true, ""product"": ""butter"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""PECHS""}"\nbhai aaj shaam 10 litre doodh bhej dena DHA ph 6,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""today evening"", ""area"": ""DHA Phase 6""}"\n3 kg makhan bhejwa dein Bahadurabad please,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": null, ""area"": ""Bahadurabad""}"\n"dha phase6 wala order: 3 kilo desi ghee, Friday ko","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""DHA Phase 6""}"\nplz 3 liter doodh DHA ph 6,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 3, ""unit"": ""litre"", ""delivery_time"": null, ""area"": ""DHA Phase 6""}"\n"parson ke liye 2 kilo desi ghee book kar lein, N. Nazimabad","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""North Nazimabad""}"\nbhai rider 1 ghanta late aaya,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nbhai kal shaam 3 kilo makkhan bhej dena clifton block 5,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": ""Clifton Block 5""}"\n"mujhe lassi chahiye abhi, north nazimabad","{""is_order"": true, ""product"": ""lassi"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""now"", ""area"": ""North Nazimabad""}"\nAoA bhai 2 kilo desi ghee Friday ko chahiye. address DHA ph 6,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""DHA Phase 6""}"\nlassi chahiye 10 litre Sunday subah,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""Sunday morning"", ""area"": null}"\n"Assalam o alaikum, pechs mein 3 kg dahi chahiye parson","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""PECHS""}"\n"Assalam o alaikum, Gulshan e Iqbal mein 2 kilo ghee chahiye Friday ko","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Gulshan-e-Iqbal""}"\n"kal subah ke liye 5 kg ghee book kar lein, Gulshan e Iqbal","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Gulshan-e-Iqbal""}"\nbhai kal shaam 1 kilo makkhan bhej dena Gulshan e Iqbal,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": ""Gulshan-e-Iqbal""}"\nPlease call me back,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nkya aap paneer bhi bechte hain?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nplz 1 kilo desi ghee north nazimabad,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": null, ""area"": ""North Nazimabad""}"\n"dha phase 6 wala order: aadha kilo makkhan, parson","{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""DHA Phase 6""}"\nAoA bhai 3 litre dudh kal subah chahiye. address DHA ph 6,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 3, ""unit"": ""litre"", ""delivery_time"": ""tomorrow morning"", ""area"": ""DHA Phase 6""}"\n"kal subah aadha kilo desi ghee rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": null}"\n"1 kg butter please, deliver tomorrow morning to Bahadurabad","{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Bahadurabad""}"\nplz 2 kg ghee N. Nazimabad,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": null, ""area"": ""North Nazimabad""}"\n"Assalam o alaikum, dha phase 6 mein 1 liter dudh chahiye Sunday subah","{""is_order"": true, ""product"": ""milk"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""Sunday morning"", ""area"": ""DHA Phase 6""}"\nbhai parson 2 litre lassi bhej dena dha phase6,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 2, ""unit"": ""litre"", ""delivery_time"": ""day after tomorrow"", ""area"": ""DHA Phase 6""}"\nghee pure hai ya mix?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nplz 2 kilo makkhan gulshan,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": null, ""area"": ""Gulshan-e-Iqbal""}"\ndoodh 2 liter abhi tak dha phase 6 pohncha dein,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 2, ""unit"": ""litre"", ""delivery_time"": ""now"", ""area"": ""DHA Phase 6""}"\nbhai kal subah 1 kg makhan bhej dena bahadurabad,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Bahadurabad""}"\nbhai abhi 5 ltr doodh bhej dena dha phase6,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 5, ""unit"": ""litre"", ""delivery_time"": ""now"", ""area"": ""DHA Phase 6""}"\n"clifton block 5 wala order: aadha kilo dahi, Friday ko","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Clifton Block 5""}"\n"Hi, I need 1 liters fresh milk tomorrow evening at N. Nazimabad","{""is_order"": true, ""product"": ""milk"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""tomorrow evening"", ""area"": ""North Nazimabad""}"\n"Sunday subah 5 kg makkhan rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""butter"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""Sunday morning"", ""area"": null}"\nbhai Sunday subah 4 liter lassi bhej dena clifton block 5,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": ""Sunday morning"", ""area"": ""Clifton Block 5""}"\n"Friday ko 1 litre lassi rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""lassi"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""Friday"", ""area"": null}"\n"Hi, I need 4 liters milk this evening at PECHS","{""is_order"": true, ""product"": ""milk"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": ""today evening"", ""area"": ""PECHS""}"\n"Hi, I need 2 kg pure ghee tomorrow morning at bahadurabad","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Bahadurabad""}"\n"Clifton blk 5 wala order: 2 kilo makhan, aaj shaam","{""is_order"": true, ""product"": ""butter"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""Clifton Block 5""}"\n"Friday ko ke liye 1 kg desi ghee book kar lein, bahadurabad","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""Bahadurabad""}"\n"mujhe makhan chahiye Sunday subah, bahadurabad","{""is_order"": true, ""product"": ""butter"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""Sunday morning"", ""area"": ""Bahadurabad""}"\nplz 4 ltr doodh dha phase 6,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": null, ""area"": ""DHA Phase 6""}"\nkal wala order cancel kar dein,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nmakkhan 5 kilo parson tak dha phase 6 pohncha dein,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""DHA Phase 6""}"\nbhai kal subah 10 litre lassi bhej dena gulshan,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Gulshan-e-Iqbal""}"\nmakhan aadha kilo kal subah tak north nazimabad pohncha dein,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""North Nazimabad""}"\n"Assalam o alaikum, PECHS mein 5 kilo desi ghee chahiye aaj shaam","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""PECHS""}"\n"Assalam o alaikum, dha phase 6 mein 3 kg ghee chahiye parson","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""DHA Phase 6""}"\n"pechs wala order: 1 kilo makkhan, kal shaam","{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": ""PECHS""}"\nAoA bhai 1 kg makhan Friday ko chahiye. address N. Nazimabad,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""North Nazimabad""}"\nOk,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"Hi, I need 1 kg pure ghee day after tomorrow at Gulshan e Iqbal","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": ""Gulshan-e-Iqbal""}"\nbhai Friday ko 3 liter lassi bhej dena Gulshan e Iqbal,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 3, ""unit"": ""litre"", ""delivery_time"": ""Friday"", ""area"": ""Gulshan-e-Iqbal""}"\nAssalam o alaikum,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n😊👍,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nrider ka number bhej dein,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\naaj shaam desi ghee bhej dein pechs,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""today evening"", ""area"": ""PECHS""}"\nmakkhan 2 kilo kal shaam tak clifton block 5 pohncha dein,"{""is_order"": true, ""product"": ""butter"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": ""Clifton Block 5""}"\n"Assalam o alaikum, N. Nazimabad mein aadha kilo dahi chahiye Sunday subah","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""Sunday morning"", ""area"": ""North Nazimabad""}"\n"parson 2 kilo desi ghee rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""day after tomorrow"", ""area"": null}"\n"clifton block 5 wala order: 10 ltr dudh, abhi","{""is_order"": true, ""product"": ""milk"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""now"", ""area"": ""Clifton Block 5""}"\nJazzCash se payment ho sakti hai?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"Hi, I need 10 litres lassi tomorrow morning at dha phase 6","{""is_order"": true, ""product"": ""lassi"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""tomorrow morning"", ""area"": ""DHA Phase 6""}"\nThank you for the quick delivery,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\nbhai aaj shaam 3 kilo dahi bhej dena dha phase 6,"{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 3, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""DHA Phase 6""}"\nFriday ko lassi bhej dein clifton block 5,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""Friday"", ""area"": ""Clifton Block 5""}"\nAoA bhai 1 litre lassi Friday ko chahiye. address DHA ph 6,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": 1, ""unit"": ""litre"", ""delivery_time"": ""Friday"", ""area"": ""DHA Phase 6""}"\n"Assalam o alaikum, N. Nazimabad mein 2 kg desi ghee chahiye aaj shaam","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""today evening"", ""area"": ""North Nazimabad""}"\ndesi ghee 2 kilo Friday ko tak pechs pohncha dein,"{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 2, ""unit"": ""kg"", ""delivery_time"": ""Friday"", ""area"": ""PECHS""}"\nmera address change ho gaya hai,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"Assalam o alaikum, Clifton blk 5 mein 5 kg makkhan chahiye kal subah","{""is_order"": true, ""product"": ""butter"", ""quantity"": 5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Clifton Block 5""}"\n"Clifton blk 5 wala order: aadha kilo makhan, kal shaam","{""is_order"": true, ""product"": ""butter"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": ""Clifton Block 5""}"\nlast week ka bill kitna bana?,"{""is_order"": false, ""product"": null, ""quantity"": null, ""unit"": null, ""delivery_time"": null, ""area"": null}"\n"2 litres lassi please, deliver tomorrow morning to bahadurabad","{""is_order"": true, ""product"": ""lassi"", ""quantity"": 2, ""unit"": ""litre"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Bahadurabad""}"\n"Assalam o alaikum, bahadurabad mein 10 liter dudh chahiye aaj shaam","{""is_order"": true, ""product"": ""milk"", ""quantity"": 10, ""unit"": ""litre"", ""delivery_time"": ""today evening"", ""area"": ""Bahadurabad""}"\nkal shaam lassi bhej dein clifton block 5,"{""is_order"": true, ""product"": ""lassi"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""tomorrow evening"", ""area"": ""Clifton Block 5""}"\n"1 kg yogurt please, deliver tomorrow morning to Gulshan e Iqbal","{""is_order"": true, ""product"": ""yogurt"", ""quantity"": 1, ""unit"": ""kg"", ""delivery_time"": ""tomorrow morning"", ""area"": ""Gulshan-e-Iqbal""}"\n"N. Nazimabad wala order: 5 litre dudh, aaj shaam","{""is_order"": true, ""product"": ""milk"", ""quantity"": 5, ""unit"": ""litre"", ""delivery_time"": ""today evening"", ""area"": ""North Nazimabad""}"\n"kal shaam aadha kilo desi ghee rakh dena, main khud le jaunga","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": 0.5, ""unit"": ""kg"", ""delivery_time"": ""tomorrow evening"", ""area"": null}"\nbhai abhi 2 ltr doodh bhej dena Clifton blk 5,"{""is_order"": true, ""product"": ""milk"", ""quantity"": 2, ""unit"": ""litre"", ""delivery_time"": ""now"", ""area"": ""Clifton Block 5""}"\n"mujhe desi ghee chahiye parson, DHA ph 6","{""is_order"": true, ""product"": ""desi ghee"", ""quantity"": null, ""unit"": null, ""delivery_time"": ""day after tomorrow"", ""area"": ""DHA Phase 6""}"\n"4 liters fresh milk please, deliver this evening to Gulshan e Iqbal","{""is_order"": true, ""product"": ""milk"", ""quantity"": 4, ""unit"": ""litre"", ""delivery_time"": ""today evening"", ""area"": ""Gulshan-e-Iqbal""}"\n',
    'examples/01_farm_order_slips/system_prompt.txt': 'You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only.\n',
    'examples/04_bad_dataset/bad_examples.jsonl': '{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai kal subah 5 litre doodh bhej dena DHA phase 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 5, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "2 kg dahi please, deliver this evening to clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for ..."}, {"role": "user", "content": "3 litre lassi PECHS}]}\n{"messages": [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Friday ko 1 kilo makhan chahiye, Bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "aaj shaam 2 litre doodh gulshan"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"qty\\": 2, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "parson 1 kilo desi ghee North Nazimabad"}, {"role": "assistant", "content": "Sure! Here is your order: {\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai kal subah 5 litre doodh bhej dena DHA phase 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 5, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal shaam 4 litre doodh PECHS"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "10 litre doodh kal subah, call karein 0321-4567890 DHA phase 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Can you send 5 litres of milk tomorrow morning? Address: Bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 3, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal subah 2 litre doodh dha ph 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 2, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"dha ph6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": ""}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": null, \\"area\\": null}"}]}\n',
}
# write every file
for path, text in files.items():
    # make the folder if it does not exist yet
    os.makedirs(os.path.dirname(path), exist_ok=True)
    # write the text into the file
    with open(path, "w", encoding="utf-8") as f:
        # save it
        f.write(text)
# show what we created
print("Created:", list(files))

## 👀 Step 2 — Look at the raw spreadsheet
Students write examples in **Excel / Google Sheets** with two columns: `input` (what the customer says) and `output` (the PERFECT answer).

In [ ]:
# 'csv' reads spreadsheet files
import csv
# open the farm example spreadsheet
with open("examples/01_farm_order_slips/examples.csv", encoding="utf-8") as f:
    # read rows as dictionaries using the header row
    rows = list(csv.DictReader(f))
# how many examples
print(len(rows), "examples\n")
# show the first 5
for row in rows[:5]:
    # the customer's message
    print("INPUT :", row["input"])
    # the perfect answer
    print("OUTPUT:", row["output"], "\n")

## 🔁 Step 3 — Spreadsheet → JSONL (train / val / test)

In [ ]:
!python make_dataset.py examples/01_farm_order_slips

In [ ]:
# 'json' reads one line of the JSONL file
import json
# open the training file
with open("examples/01_farm_order_slips/train.jsonl", encoding="utf-8") as f:
    # read only the first line = the first example
    first = json.loads(f.readline())
# print it nicely: system / user / assistant
print(json.dumps(first, indent=2, ensure_ascii=False))

Ab check karein: **dataset doctor** 🩺

In [ ]:
!python validate_dataset.py examples/01_farm_order_slips

## 🎮 Step 4 — Game: spot the bad dataset
`bad_examples.jsonl` mein **10 ghaltiyan** hain. Pehle script chalayein — woh kitni pakarti hai?

In [ ]:
!python validate_dataset.py examples/04_bad_dataset/bad_examples.jsonl

Script ne 8 pakri. **2 ghaltiyan sirf insaan pakar sakta hai.** Neeche lines 10 aur 11 parhein — kya ghalat hai? 🤔

In [ ]:
# read all lines of the bad file
lines = open("examples/04_bad_dataset/bad_examples.jsonl", encoding="utf-8").read().splitlines()
# look at line 10 and line 11 (Python counts from 0, so index 9 and 10)
for number in (10, 11):
    # turn the line into Python data
    msgs = json.loads(lines[number - 1])["messages"]
    # show the customer message
    print(f"LINE {number} INPUT :", msgs[1]["content"])
    # show the answer
    print(f"LINE {number} OUTPUT:", msgs[2]["content"], "\n")

<details><summary>👉 Answer (click)</summary>

Line 10: customer asked for **5** litres, answer says **3**.  
Line 11: area written `dha ph6` instead of `DHA Phase 6` (not consistent).  
**Lesson:** the script checks FORMAT, only you can check MEANING.
</details>

## 🏆 Step 5 — YOUR dataset
**Option A:** write examples directly in the cell below (good for starting).
**Option B:** upload your `examples.csv` from Google Sheets (cell after that).

First think: *Is my task narrow? Could two people write the SAME correct answer?*

In [ ]:
# ===== OPTION A: write your examples here =====
# your project folder name (no spaces)
PROJECT = "my_project"
# ONE clear instruction, the same for every example
SYSTEM_PROMPT = "Classify a customer complaint into one department: delivery, quality, billing or other. Reply with the department name only."
# your examples: (input, perfect output). Add MANY more (homework: 100+)
EXAMPLES = [
    # delivery problems
    ("rider ne galat ghar pe order de diya", "delivery"),
    ("order 2 ghante late aaya", "delivery"),
    # quality problems
    ("doodh ka taste ajeeb tha", "quality"),
    ("dahi mein paani zyada tha", "quality"),
    # billing problems
    ("bill mein 500 rupay zyada lage hain", "billing"),
    ("JazzCash se do dafa paise cut gaye", "billing"),
    # other
    ("aapka office kahan hai?", "other"),
    ("rider ka number chahiye", "other"),
    ("ghee ki packing phati hui thi", "quality"),
    ("delivery boy ne change wapas nahi diya", "billing"),
    ("subah 7 ki bajaye 10 baje aaya", "delivery"),
    ("Sunday ko khule hain?", "other"),
]

# make the folder
os.makedirs(PROJECT, exist_ok=True)
# save the system prompt
open(f"{PROJECT}/system_prompt.txt", "w", encoding="utf-8").write(SYSTEM_PROMPT)
# save the examples as a CSV with the header input,output
with open(f"{PROJECT}/examples.csv", "w", newline="", encoding="utf-8") as f:
    # a CSV writer handles commas and quotes for us
    writer = csv.writer(f)
    # header row
    writer.writerow(["input", "output"])
    # all example rows
    writer.writerows(EXAMPLES)
# confirm
print(f"Saved {len(EXAMPLES)} examples to {PROJECT}/examples.csv")

In [ ]:
# ===== OPTION B: upload your own examples.csv (skip if you used option A) =====
# Colab's upload button
from google.colab import files
# opens a file picker; choose your examples.csv
uploaded = files.upload()
# move the uploaded file into the project folder
for name, data in uploaded.items():
    # write the bytes into PROJECT/examples.csv
    open(f"{PROJECT}/examples.csv", "wb").write(data)
    # confirm
    print("Saved", name, "as", f"{PROJECT}/examples.csv")
# remember: also write SYSTEM_PROMPT in the option A cell and run it once

In [ ]:
# build train/val/test from your CSV
!python make_dataset.py {PROJECT}

In [ ]:
# check your dataset (fix errors, read warnings)
!python validate_dataset.py {PROJECT}

In [ ]:
# download your finished dataset as a zip
# 'shutil' makes zip files
import shutil
# zip the whole project folder
shutil.make_archive(PROJECT, "zip", PROJECT)
# try the Colab download (in VS Code the zip is simply saved in this folder)
try:
    # Colab download helper
    from google.colab import files
    # start the download
    files.download(f"{PROJECT}.zip")
# not on Colab
except ImportError:
    # tell the student where the file is
    print(f"Saved {PROJECT}.zip in this folder")

## ✅ Before next class
- [ ] 100+ examples (better 200+), at least 10–15% edge cases
- [ ] `validate_dataset.py` says **READY**
- [ ] You read 20 random examples yourself
- [ ] `test.jsonl` untouched — it's next week's final exam!

See `HOMEWORK.md` for all rules. Next week: **LoRA / QLoRA fine-tuning on YOUR data** 🚀

---
*Module 10 designed by Engr Usman Hasan*